# SuperEEG leave-one-out analysis

The objective is to measure the quality of the SuperEEG model. We perform a leave-one-out analysis and evaluate the quality of the reconstruction. We can measure the quality of the model by examining statistics such as average and histogrammed RMSE and metrics of individual traces.

The leave-one-out analysis looks like this:

1. load in all the data (resample to 250Hz ?)
2. cull electrodes which do not pass a kurtosis test
3. cull brains with fewer than two (three?) remaining electrodes
4. compile all remaining electrode locations
5. for each brain:
    1. compute full-brain collelation matrix K using all other patients' data (build models from brains using locs, then build one model from those models)
    2. for each electrode in this brain:
        1. extract Y_ska by removing this electrode
        2. compute Y_skb = (K_ba*inv(K_aa) * Y_ska.T).T
        3. compare Y_skb with observed trace
6. do something with the predicted traces. correlation

### Load in the data

let's start off with a small dataset - DANDI 000576. after loading them, compile the electrode locations and make brain objects with those locations

In [2]:
import supereeg as se
import os
import numpy as np
from scipy.io import loadmat
from pynwb import NWBHDF5IO
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import multiprocessing as mp
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

In [3]:
# needs updated to handle multiple sessions
def load_Miller():
    task = 'faceshouses-basic'
    data_dir = f'../brain-lab-data/miller-BIDS/{task}'
    data_name = f'{task}_ieeg.mat'
    locs_name = 'electrodes.mat'
    
    # read the subject ids from the directory
    IDS = [x[-2:] for x in os.listdir(data_dir) if x.startswith('sub-') and 'jm' not in x]
    
    # read in the data
    DATA = {f'{x}': loadmat(f"{data_dir}/sub-{x}/ieeg/sub-{x}_{data_name}") for x in IDS}
    
    # read in the electrode locations
    for x in IDS:
        DATA[x]['locs'] = loadmat(f"{data_dir}/sub-{x}/ieeg/sub-{x}_{locs_name}")['locs']
    
    # compile all locations
    R = np.unique(np.concatenate([DATA[x]['locs'] for x in IDS]),axis=0)
    
    # make brain objects for each subject
    BOS = {f'{x}': se.Brain(data = DATA[x]['data'], locs = DATA[x]['locs'], 
                    sample_rate = 1000) for x in IDS}

    return IDS, DATA, R, BOS

In [21]:
def load_DANDI(data_dir, read_cmd):
    
    # read the subject ids from the directory
    IDS = [x.split("-")[1] for x in os.listdir(data_dir) if x.startswith('sub-')]

    # read in the data
    DATA = {}
    pop_list = []
    for x in IDS:
        
        # make data structure
        DATA[x] = {}

        # iterate over sessions
        for file in os.listdir(sorted(Path(f'{data_dir}').glob(f'sub-{x}*'))[0]):
            if file.endswith('GP31-B1.nwb'):
                
                # get the session name
                sess = file.split("_")[1].split("-")[1]
                try:
                    if sess in DATA[x]['sessions']: continue
                except:
                    pass
                
                # open the file with HDPy
                f = NWBHDF5IO(sorted(Path(f'{data_dir}').glob(f'sub-{x}*/*{sess}*.nwb'))[0], "r").read()
                
                # read from file
                try:
                    # get electrode locations
                    # ASSUMING ALL SESSIONS HAVE SAME ELECTRODES
                    trodes = f.electrodes[:][['x','y','z']].to_numpy()
                    if np.any(np.isnan(trodes)): raise Exception()
                    DATA[x]['locs'] = trodes
                    print(f'session {f.identifier} has locs')
                    
                    # get electrode readings
                    try:
                        DATA[x]['data'] = np.vstack((DATA[x]['data'], eval(read_cmd)))
                    except:
                        DATA[x]['data'] = eval(read_cmd).astype(float)
                        print(DATA[x]['data'].shape)
                        N = DATA[x]['data'].shape[0]
                    
                    # save session name
                    try:
                        DATA[x]['sessions'] += [sess]*N
                    except:
                        DATA[x]['sessions'] = [sess]*N
                
                except:
                    print(f'locs not available for {f.identifier}')
                    pop_list.append(x)
    
    # remove IDS of datasets with no locations
    for item in pop_list:
        if item in IDS: IDS.remove(item)
    
    # compile all locations
    R = np.unique(np.vstack([DATA[x]['locs'] for x in IDS]))
    
    # make brain objects for each subject
    # ASSUMING ALL SESSIONS ARE THE SAME LENGTH
    BOS = {f'{x}': se.Brain(data = DATA[x]['data'], locs = DATA[x]['locs'], 
            sessions = DATA[x]['sessions'], sample_rate = 2000) for x in IDS}
    
    return IDS, DATA, R, BOS

In [22]:
IDS, DATA, R, BOS  = load_DANDI('../brain-lab-data/DANDI/000019', 
                               'f.acquisition["ElectricalSeries"].data[:]')

session GP31_B1 has locs
(1226807, 256)


In [5]:
data_dir = '../brain-lab-data/DANDI/000019'
x = 'GP31'
sess = 'GP31-B1'
f = NWBHDF5IO(sorted(Path(f'{data_dir}').glob(f'sub-{x}*/*{sess}*.nwb'))[0], "r").read()
f

root pynwb.file.NWBFile at 0x140613715627088
Fields:
  acquisition: {
    ElectricalSeries <class 'pynwb.ecephys.ElectricalSeries'>
  }
  devices: {
    L256Grid <class 'pynwb.device.Device'>
  }
  electrode_groups: {
    L256Grid electrodes <class 'pynwb.ecephys.ElectrodeGroup'>
  }
  electrodes: electrodes <class 'hdmf.common.table.DynamicTable'>
  epochs: epochs <class 'pynwb.epoch.TimeIntervals'>
  file_create_date: [datetime.datetime(2019, 6, 19, 12, 38, 44, 781147, tzinfo=tzoffset(None, -25200))]
  identifier: GP31_B1
  institution: University of California, San Francisco
  intervals: {
    epochs <class 'pynwb.epoch.TimeIntervals'>,
    trials <class 'pynwb.epoch.TimeIntervals'>
  }
  lab: Chang Lab
  session_description: GP31_B1
  session_id: GP31_B1
  session_start_time: 1900-01-01 08:00:00+00:00
  subject: subject pynwb.file.Subject at 0x140617035427392
Fields:
  species: Homo sapiens
  subject_id: GP31

  timestamps_reference_time: 1900-01-01 08:00:00+00:00
  trials: trials <class 'pynwb.epoch.TimeIntervals'>

### Kurtosis threshold

trim out the electrodes that don't pass the test. cull the brains that don't have enough electrodes

the Miller data don't include kurtosis and i think they're already trimmed

In [ ]:
# put something here when there's data that need it

### For each brain

compute fbcm excluding subject brain

### For each electrode

estimate activity and compare

multiprocessing architecture:
1. pool1 of 8 for subjects
    1. pool2 of 2 for trodes
    2. save results to file as they are completed
    3. close/join pool2
2. close/join pool1

### Make an output folder

In [10]:
try:
    out_dir = f'{data_dir}/out'
    os.mkdir(out_dir)
except:
    print('out directory exists')

### multi-threaded

In [29]:
%%time
def f1(sub):
    # make a model using all the brains except the subject at all locations
    mo = se.Model([BOS[x] for x in IDS if x != sub], locs=R)
    
    # get the correlation matrix
    K = mo.get_model()

    # get subject electrode locations
    locs = DATA[sub]['locs']

    def f2(i,trode):
        # get Y_ska by removing this electrode
        Y_ska = DATA[sub]['data'][:,np.where(~np.any(locs != trode, axis=1))[0]]

        # get the indices of R where this patient's electrodes are
        aa_mask = np.array([True if np.any(np.all(x == locs, axis=1)) and (x != trode).any() else False for x in R])
        ba_mask = np.array([True if (x == trode).all() else False for x in R])

        # get K_aa and K_ba using these masks
        K_aa = K[np.ix_(aa_mask,aa_mask)]
        K_ba = K[np.ix_(ba_mask,aa_mask)]
        
        # compute Y_skb
        Y_skb = ((K_ba@np.linalg.inv(K_aa))@(Y_ska.T)).T

        # save predicted trace
        np.savetxt(f'{out_dir}/sub-{sub}_electrode-{i}_recon.csv', Y_skb)

    # iterate over the electrodes in this subject's brain
    for i,trode in enumerate(locs):
        f2(i,trode)

# iterate over the subjects
with mp.Pool(8) as p1:
    p1.map(f1, IDS)
    p1.close()
    p1.join()

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 1 is different from 255)

### single-threaded

In [ ]:
%%time
# iterate over the subjects
for sub in IDS:
    # make a model using all the brains except the subject at all locations
    mo = se.Model([BOS[x] for x in IDS if x != sub], locs=R)
    
    # get the correlation matrix
    K = mo.get_model()

    # get subject electrode locations
    locs = DATA[sub]['locs']

    # iterate over the electrodes in this subject's brain
    for i,trode in enumerate(locs):
        
        # get Y_ska by removing this electrode
        Y_ska = DATA[sub]['data'][:,np.unique(np.where(~(locs == trode))[0])]

        # get the indices of R where this patient's electrodes are
        aa_mask = np.array([True if np.any(np.all(x == locs, axis=1)) and (x != trode).any() else False for x in R])
        ba_mask = np.array([True if (x == trode).all() else False for x in R])
        
        # get K_aa and K_ba using these masks
        K_aa = K[np.ix_(aa_mask,aa_mask)]
        K_ba = K[np.ix_(ba_mask,aa_mask)]

        # compute Y_skb
        Y_skb = ((K_ba@np.linalg.inv(K_aa))@(Y_ska.T)).T

        # save predicted trace
        np.savetxt(f'{out_dir}/sub-{sub}_electrode-{i}_recon.csv', Y_skb)